# `WorldBuilder` with real OpenAI

End-to-end use of `src/llm/world_builder.WorldBuilder` against the live `OpenAIClient`. Covers:

1. Setup + cost expectations
2. **Method A** — `build()` one-shot (four LLM calls, returns a full `World`)
3. **Method B** — calling each stage individually for finer control / inspection
4. Effective-use patterns: caching, retries, archetype phrasing
5. **Generate a 100-item realistic catalog** and persist it

**Requires** `OPENAI_API_KEY` in the environment. Default model: `gpt-4o-2024-08-06`.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import os
import sys
from collections import Counter
from dataclasses import asdict, fields
from pathlib import Path

sys.path.append('../')

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY before running this notebook'

In [ ]:
from src.llm.openai_client import OpenAIClient
from src.llm.world_builder import World, WorldBuilder, allocate_skeletons, _MARKET_MATH_DEFAULTS

# Using gpt-4o-mini for cost. Swap to OpenAIClient() for the default
# gpt-4o-2024-08-06, which produces higher-quality but pricier output.
# client = OpenAIClient(model='gpt-4o-mini')
client = OpenAIClient(model='gpt-5.4-mini')
# client = OpenAIClient(model='gpt-5.4')


## Cost expectations

`build(n_items)` issues exactly **4 LLM calls**:
1. `MarketDomain` — small payload (~regions, cycle params, elasticity)
2. `StoreTemplateList` — small payload (one row per region/tier)
3. `Taxonomy` — small payload (3-8 categories)
4. `Catalog` — scales with `n_items` (the dominant cost)

Stages are cached on the builder instance, so repeat `build()` calls do not re-bill. Spawn a fresh `WorldBuilder` to start over.

## Method A — `build()` one-shot

Cleanest path: pick an archetype, call `build(n_items)`, get a `World(catalog, market, store_templates)`.

In [ ]:
ARCHETYPE = 'fashion_retail'

builder_a = WorldBuilder(archetype=ARCHETYPE, client=client)
world = builder_a.build(n_items=12)

print(f'catalog        : {len(world.catalog)} items')
print(f'templates      : {list(world.store_templates.keys())}')
print(f'market.regions : {world.market.regions}')
print(f'cycle/peak/off : {world.market.cycle_len} / {world.market.peak_factor} / {world.market.off_factor}')

In [ ]:
catalog_df = world.catalog_df()
catalog_df['margin_pct'] = (catalog_df['margin'] / catalog_df['base_price']).round(3)
catalog_df


In [ ]:
world.store_templates_df().set_index('id')


In [ ]:
# Re-calling build() does NOT hit the LLM again — every stage is cached.
world_again = builder_a.build(n_items=12)
print('cached:', world_again.catalog is world.catalog)

## Method B — call each stage individually

Use this when you want to:
- inspect the taxonomy before paying for a 100-item catalog
- tweak skeleton allocation (e.g., upsample a category)
- regenerate just one stage (drop the cache by spawning a new builder)

Stages are independent functions — they don't have to run in `build()` order, but `build_store_templates()` will trigger `build_market_domain_params()` if regions aren't yet known.

In [ ]:
builder_b = WorldBuilder(archetype=ARCHETYPE, client=client)

### B1. `build_taxonomy()` — 1 LLM call

In [ ]:
taxonomy = builder_b.build_taxonomy()
pd.DataFrame([
    {'name': c.name, 'target_share': c.target_share, 'description': c.description}
    for c in taxonomy.categories
])

### B2. `allocate_skeletons(n, taxonomy)` — deterministic, no LLM

Run this as many times as you want at zero cost. Useful for sanity-checking the category mix before paying for the catalog call.

In [ ]:
rows = []
for n in [10, 25, 50, 100, 250]:
    s = allocate_skeletons(n, taxonomy)
    rows.append({'n': n, **Counter(s)})
pd.DataFrame(rows).fillna(0).astype({c.name: int for c in taxonomy.categories}).set_index('n')

### B3. `build_market_domain_params()` — 1 LLM call

LLM authors the domain-meaningful slice; the math defaults are merged in by `WorldBuilder`.

In [ ]:
market = builder_b.build_market_domain_params()

llm_keys = {'cycle_len', 'peak_factor', 'off_factor', 'init_demand', 'init_supply',
            'season_months', 'regions', 'price_elasticity'}
rows = []
for k, v in asdict(market).items():
    rows.append({
        'field': k,
        'authored_by': 'LLM' if k in llm_keys else 'math default',
        'value': repr(v),
    })
pd.DataFrame(rows)

### B4. `build_store_templates()` — 1 LLM call

Reads `regions` from the cached market params.

In [ ]:
templates = builder_b.build_store_templates()
pd.DataFrame([
    {f.name: getattr(t, f.name) for f in fields(t)}
    for t in templates.values()
]).set_index('id')

### B5. `sample_catalog(n)` — 1 LLM call

Two-stage internally: `allocate_skeletons` (Python) + LLM naming/pricing/cross-refs. Output is `list[Ware]` with stable `P{i:04d}` ids assigned by `load_catalog`.

In [ ]:
small_catalog = builder_b.sample_catalog(8)
pd.DataFrame(small_catalog)


## Effective use

**1. One builder instance per intended world.** Stages cache. Don't share a builder across unrelated archetypes.

**2. Iterate on the taxonomy before paying for the catalog.** `build_taxonomy()` is small; `sample_catalog(100)` is the expensive call. If categories look wrong, spawn a fresh builder rather than fighting the cache.

**3. Tune `max_retries`.** On `ValidationError`, the next prompt prepends the error so the model self-corrects. Default is 3. Drop to `1` to fail fast during prompt iteration; raise to `5+` for flaky payloads.

**4. Archetype phrasing matters.** `"fashion_retail"`, `"high-end fashion boutique in northern europe"`, and `"fast fashion mass market"` will produce visibly different catalogs and store templates. The string is interpolated verbatim into every prompt.

**5. Validation contract** (enforced in `src/llm/schemas.py`): `base_price > unit_cost`, `unit_cost >= 0`, `related_products` resolve in-catalog, no self-refs, `correlation in [0, 1]`, `target_share in (0, 1]`, `price_elasticity < 0`, unique template ids.

**6. Test seam.** Anything implementing `LLMClient` Protocol (one method: `structured_completion(*, system, user, schema)`) plugs in. Useful for fixtures or mocks; see `tests/llm/test_world_builder.py`.

In [ ]:
# Example of (4): contrast two archetype phrasings on taxonomy alone (cheap).
phrasings = [
    'fast_fashion',
    'luxury fashion boutique, scandinavian minimalist',
]
for arch in phrasings:
    tax = WorldBuilder(archetype=arch, client=client).build_taxonomy()
    print(f'\n[{arch}]')
    for c in tax.categories:
        print(f'  {c.target_share:>5.2f}  {c.name}')

## Generate 100 realistic samples

Single `sample_catalog(100)` call. The model places items in skeleton order and `load_catalog` assigns `P0000`-`P0099` ids.

*Tip:* if a 100-item Pydantic payload occasionally trips a validator, the builder's retry path feeds the error back into the next prompt automatically. If you want deterministic chunking instead, see the chunked-flow cell at the end.

In [ ]:
BIG_ARCHETYPE = 'fashion_retail'
N_SAMPLES = 1000

big_builder = WorldBuilder(archetype=BIG_ARCHETYPE, client=client, max_retries=5)
big_world = big_builder.build(n_items=N_SAMPLES)

big_catalog_df = big_world.catalog_df()
big_catalog_df['margin_pct'] = (big_catalog_df['margin'] / big_catalog_df['base_price']).round(3)
print(f'catalog size: {len(big_catalog_df)}')
big_catalog_df.head(10)


In [ ]:
big_catalog_df.related_products.value_counts()

In [ ]:
# Category mix vs taxonomy targets
tax = big_builder.build_taxonomy()
target = pd.Series({c.name: c.target_share for c in tax.categories}, name='target_share')
actual = (big_catalog_df['category'].value_counts(normalize=True)
          .rename('actual_share').round(3))
pd.concat([target, actual], axis=1).fillna(0).sort_values('target_share', ascending=False)

In [ ]:
# Distributional sanity checks
summary = big_catalog_df.groupby('category').agg(
    n=('product_id', 'count'),
    avg_price=('base_price', 'mean'),
    avg_cost=('unit_cost', 'mean'),
    avg_margin_pct=('margin_pct', 'mean'),
    min_price=('base_price', 'min'),
    max_price=('base_price', 'max'),
).round(2)
summary

In [ ]:
# Cross-product correlations
rel_rows = []
for w in big_world.catalog:
    for partner_name, corr in w.related_products:
        rel_rows.append({
            'product': w.name, 'category': w.category,
            'related_to': partner_name, 'correlation': corr,
        })
rel_df = pd.DataFrame(rel_rows)
print(f'{len(rel_df)} cross-product links across {rel_df["product"].nunique() if len(rel_df) else 0} products')
rel_df.head(15) if len(rel_df) else 'no related_products links generated'

In [ ]:
# Seasonality breakdown
big_catalog_df['seasonality'].value_counts()

## (Optional) Chunked sampling for very large catalogs

If you push beyond a few hundred items in one call, structured-output payloads can get unwieldy. Pattern below: run the taxonomy once, then call `sample_catalog` with a batch size in a fresh builder seeded with the same taxonomy. Note: ids restart at `P0000` per builder, so you must re-id manually if you concatenate catalogs.

In [ ]:
# Sketch only - uncomment to run.
# from src.sim.scenario import Ware
#
# CHUNK = 25
# TOTAL = 100
# combined = []
# for i in range(0, TOTAL, CHUNK):
#     b = WorldBuilder(archetype=BIG_ARCHETYPE, client=client)
#     b._taxonomy = tax  # seed cached taxonomy so allocator stays consistent
#     part = b.sample_catalog(min(CHUNK, TOTAL - i))
#     combined.extend(part)
# # Re-id so P0000..P{TOTAL-1} are stable across the concatenation.
# combined = [w._replace(product_id=f'P{i:04d}') for i, w in enumerate(combined)]
# pd.DataFrame(combined).head()


In [ ]:
# Validate related_products references in big_catalog_df
# Rules:
# 1) every related product name must exist in this catalog
# 2) a product cannot reference itself by name

product_names = set(big_catalog_df['name'])

invalid_rows = []
for idx, row in big_catalog_df[['product_id', 'name', 'related_products']].iterrows():
    pid = row['product_id']
    product_name = row['name']
    rels = row['related_products']

    if not isinstance(rels, list):
        continue

    missing_refs = []
    self_refs = []

    for rel in rels:
        # related_products are expected as (related_product_name, correlation)
        if isinstance(rel, (list, tuple)) and len(rel) >= 1:
            rel_name = rel[0]
        elif isinstance(rel, dict):
            rel_name = rel.get('name') or rel.get('related_name') or rel.get('related_product_name')
        else:
            rel_name = rel

        if rel_name == product_name:
            self_refs.append(rel_name)
        elif rel_name not in product_names:
            missing_refs.append(rel_name)

    if missing_refs or self_refs:
        invalid_rows.append({
            'row_index': idx,
            'product_id': pid,
            'product_name': product_name,
            'missing_related_names': missing_refs,
            'self_references_by_name': self_refs,
            'raw_related_products': rels,
        })

issues_df = pd.DataFrame(invalid_rows)

if issues_df.empty:
    print('All good: every related product name exists and no product references itself.')
else:
    print(f'Found {len(issues_df)} rows with related_products issues.')
    display(issues_df.sort_values(['product_name', 'row_index']).reset_index(drop=True))

### Persist the world

Save `world.to_json()` to `data/worlds/<name>/world.json` so subsequent scenario
scripts can call `load_or_build_world(name, build_fn)` and get an instant cache hit
instead of re-issuing LLM calls.


In [ ]:
world_name = f"{BIG_ARCHETYPE}_{N_SAMPLES}"
world_path = f"../data/worlds/{world_name}/world.json"
big_world.to_json(world_path)
print(f"Saved world to {world_path}")
print(f"Reload via: World.from_json('{world_path}')")
